In [1]:
import sys
import torch
import pandas as pd
import numpy as np
import torch.nn as nn
from torch import optim
from pathlib import Path
from torch.utils.data import DataLoader, Dataset

In [2]:
cwd = Path.cwd()
project_root = cwd.parent
data_path = project_root / "data" / "archive" / "data.csv"
labels_path = project_root / "data" / "archive" / "labels.csv"

In [3]:
data = pd.read_csv(
    filepath_or_buffer=str(data_path),
    encoding='latin-1',
    sep=",",
    thousands=',',
    na_values=['NA', 'N/A', 'null', 'NULL', '', ' ', 'None'],
)

labels = pd.read_csv(
    filepath_or_buffer=str(labels_path)
)

In [4]:
data.drop(columns=['Unnamed: 0'], inplace=True)

In [5]:
data.shape

(801, 20531)

In [6]:
vc = labels['Class'].value_counts()

In [7]:
b_data = data.iloc[labels[(labels['Class'] == vc.index[0]) | (labels['Class'] == vc.index[1])].index]

In [8]:
b_labels = labels.iloc[labels[(labels['Class'] == vc.index[0]) | (labels['Class'] == vc.index[1])].index]

In [9]:
true_labels = np.where(b_labels["Class"].astype(str) == "BRCA", 1, 0)

In [10]:
X = b_data.values

In [11]:
std = X.std(axis=0, keepdims=True)
std[std == 0] = 1
X = (X - X.mean(axis=0, keepdims=True)) / std

In [12]:
## Defining the device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

device

device(type='mps')

### Defining The Network

In [13]:
class FFN(nn.Module):
    def __init__(self, input_size: int):
        super().__init__()

        self.relu = nn.ReLU()
        self.layer1 = nn.Linear(in_features=input_size, out_features=512)
        self.layer2 = nn.Linear(in_features=512, out_features=128)
        self.layer3 = nn.Linear(in_features=128, out_features=32)
        self.layer4 = nn.Linear(in_features=32, out_features=1)

    def forward(self, X):           # (batch_size, 20531)
        X = self.layer1(X)          # (batch_size, 512)
        X = self.relu(X)            # (batch_size, 512)
        X = self.layer2(X)          # (batch_size, 128)
        X = self.relu(X)            # (batch_size, 128)
        X = self.layer3(X)          # (batch_size, 32)
        X = self.relu(X)            # (batch_size, 32)
        X = self.layer4(X)          # (batch_size, 1)
        
        return X

In [14]:
class MyDataset(Dataset):
    def __init__(self, X, y):
        super().__init__()
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, index):
        x = self.X[index]
        y = self.y[index]
        return x, y

In [15]:
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(true_labels, dtype=torch.float32)

In [16]:
train_dataset = MyDataset(X, y)
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)

### Let's Initialize model, optimizers and hyperparameters

In [24]:
LEARNING_RATE = 3e-2
EPOCHS = 10
INPUT_SIZE = train_dataset.X.shape[1]
ffn = FFN(input_size=INPUT_SIZE).to(device)
criterion = nn.BCEWithLogitsLoss()
opt = optim.Adam(params=ffn.parameters(), lr=LEARNING_RATE)

In [25]:
def train_ffn(
    dataloader: DataLoader,
    ffn: FFN,
    opt: optim.Adam,
    epochs: int,
    criterion: nn.BCEWithLogitsLoss,
    device
):
    training_cost_track = []
    for epoch in range(epochs):
        training_cost = 0
        for batch, labels in dataloader:
            batch = batch.to(device)
            labels = labels.to(device).unsqueeze(1)

            # forward_pass
            logits = ffn(batch)
            cost = criterion(logits, labels)
            training_cost += cost.item()

            # backward_pass
            opt.zero_grad()
            cost.backward()
            opt.step()

        avg_batch_error = training_cost/len(dataloader)
        training_cost_track.append(avg_batch_error)
        print(f"epoch: {epoch+1}/{epochs} | avg batch cost: {avg_batch_error:.4f}")
    
    return ffn, training_cost_track

In [26]:
trained_ffn, cost_track = train_ffn(
    dataloader=train_dataloader,
    ffn=ffn,
    opt=opt,
    epochs=EPOCHS,
    criterion=criterion,
    device=device
)

epoch: 1/10 | avg batch cost: 118.5844
epoch: 2/10 | avg batch cost: 4.2338
epoch: 3/10 | avg batch cost: 1.5241
epoch: 4/10 | avg batch cost: 0.8067
epoch: 5/10 | avg batch cost: 0.7235
epoch: 6/10 | avg batch cost: 0.0000
epoch: 7/10 | avg batch cost: 0.0000
epoch: 8/10 | avg batch cost: 0.0000
epoch: 9/10 | avg batch cost: 0.0000
epoch: 10/10 | avg batch cost: 0.0000


In [27]:
num_of_parameters = sum(p.numel() for p in trained_ffn.parameters())

In [28]:
num_of_parameters

10582209

In [30]:
logits = trained_ffn(X.to(device))

In [40]:
preds = (torch.sigmoid(logits) > 0.5).int().squeeze()

In [44]:
preds.device

device(type='mps', index=0)

In [48]:
y = y.to(device)

In [50]:
accuracy = (preds == y).float().mean() * 100

In [51]:
accuracy

tensor(100., device='mps:0')